# CS 301 · Unit I notebook

Name: Dylan Reitsma · UCID: dhr · Section: 001

One notebook for the whole unit, built task by task. Each task's cells are under its own heading, and later tasks use what earlier ones define.

In [22]:
import itertools
import numpy as np

## Task 1.1 — Four gates

`check(gate, table)` runs a gate-like object on every row of a truth table. It is used by every task in this notebook.

In [23]:
def check(gate, table, name="gate"):
    """Run a gate-like object on every row of a truth table.
    Each row is (input_1, ..., input_n, expected_output)."""
    for *inputs, expected in table:
        got = gate(*inputs)
        assert got == expected, f"{name}{tuple(inputs)} gave {got}, should be {expected}"
    print(f"{name}: all {len(table)} rows correct")

TABLES = {
    "AND":  [(0, 0, 0), (0, 1, 0), (1, 0, 0), (1, 1, 1)],
    "OR":   [(0, 0, 0), (0, 1, 1), (1, 0, 1), (1, 1, 1)],
    "NAND": [(0, 0, 1), (0, 1, 1), (1, 0, 1), (1, 1, 0)],
    "NOT":  [(0, 1), (1, 0)],
}

In [24]:
'''
Now the gates themselves. Each class takes inputs that are 0 or 1 and returns 0 or 1,
following the tables in §2.3 and §2.4 of the notes. The AndGate in §2.7 of the notes
is the model; write the other three the same way.
'''

class AndGate:
    def __call__(self, a, b):
        return 1 if a and b else 0


class OrGate:
    def __call__(self, a, b):
        return 1 if a or b else 0

class NotGate:
    def __call__(self, a):
        return not a


class NandGate:
    def __call__(self, a, b):
        return 0 if a and b else 1


check(AndGate(),  TABLES["AND"],  "AND")
check(OrGate(),   TABLES["OR"],   "OR")
check(NotGate(),  TABLES["NOT"],  "NOT")
check(NandGate(), TABLES["NAND"], "NAND")

AND: all 4 rows correct
OR: all 4 rows correct
NOT: all 2 rows correct
NAND: all 4 rows correct


## Task 1.2 — OR from three NANDs

The circuit of Example 2.1, as code. Only NandGate is used.

In [21]:
def or_from_nands(a, b):
    """OR(a, b), computed by three NAND gates and nothing else (Example 2.1)."""
    nand = NandGate()
    n1 = nand(a,a)          # the top gate: both of its inputs are a
    n2 = nand(b,b)          # the bottom gate: both of its inputs are b
    return nand(n1,n2)        # the third gate, whose inputs are n1 and n2

check(or_from_nands, TABLES["OR"], "OR from three NANDs")

OR from three NANDs: all 4 rows correct


## Task 1.3 — The threshold unit

`ThresholdUnit(w, theta)` outputs 1 when w₁x₁ + … + wₙxₙ ≥ theta, else 0 (notes §3.3). It is called like a gate, so `check` applies to it. Verified on Example 3.1.

In [ ]:
class ThresholdUnit:
    """A unit with fixed weights w and threshold theta (notes, §3.3).
    Called with the inputs as separate arguments: unit(x1, x2, ...)."""

    def __init__(self, w, theta):
        self.w = tuple(w)
        self.theta = theta

    def score(self, *x):
        """The sum w1*x1 + ... + wn*xn, as one number. np.dot(self.w, x) computes it."""
        return np.dot(self.w, x)

    def __call__(self, *x):
        """1 if the score is greater than or equal to theta, else 0."""
        return 1 if self.score(*x) >= self.theta else 0


# the unit of Example 3.1, on the four inputs the example works out
EXAMPLE_31 = [(1, 0, 1, 1), (1, 1, 1, 1), (0, 1, 1, 0), (1, 1, 0, 0)]
check(ThresholdUnit((2, -1, 2), 3), EXAMPLE_31, "Example 3.1 unit")

AssertionError: Example 3.1 unit(0, 1, 1) gave 1, should be 0

In [ ]:
ThresholdUnit.b = property(lambda self: -self.theta)      # the bias, b = -theta

def fires_with_bias(unit, *x):
    """The same decision as unit(*x), written as: score + b >= 0."""
    return ...                                            # you write this line

u = ThresholdUnit((2, -1, 2), 3)
for x in itertools.product([0, 1], repeat=3):
    assert fires_with_bias(u, *x) == u(*x), f"the two forms disagree on {x}"
print("score + bias agrees with the threshold form on all 8 inputs")

## Task 1.4 — Gates by hand

A ThresholdUnit for each gate, with weights and a threshold chosen by hand, verified on the gate's truth table.

In [ ]:
HAND = {
    "AND":  ThresholdUnit((..., ...), ...),
    "OR":   ThresholdUnit((..., ...), ...),
    "NAND": ThresholdUnit((..., ...), ...),
    "NOT":  ThresholdUnit((...,), ...),        # one input, one weight
}

for name, unit in HAND.items():
    check(unit, TABLES[name], f"{name} by hand {unit.w}, theta={unit.theta}")

### How I chose them

**AND.** (what you started from, which inputs were wrong, how you changed the numbers)

**OR.** ...

**NAND.** ...

**NOT.** ...

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.2))
for ax, name in zip(axes, ["AND", "OR", "NAND"]):
    unit = HAND[name]
    (w1, w2), theta = unit.w, unit.theta
    for a, b, expected in TABLES[name]:
        ax.plot(a, b, "o", ms=9, mfc=("#CC0033" if expected else "white"), mec="#1A1A1A")
    xs = np.linspace(-0.3, 1.3, 50)
    if w2 != 0:
        ax.plot(xs, (theta - w1 * xs) / w2, color="#CC0033")   # the line w1*x1 + w2*x2 = theta
    else:
        ax.axvline(theta / w1, color="#CC0033")
    ax.set(xlim=(-0.3, 1.3), ylim=(-0.3, 1.3), xticks=[0, 1], yticks=[0, 1],
           title=f"{name}  w=({w1}, {w2}), θ={theta}")
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Task 1.5 — A three-input AND

The same kind of unit with three weights. Eight rows.

In [ ]:
AND3 = ThresholdUnit((..., ..., ...), ...)

AND3_TABLE = [(*x, int(all(x))) for x in itertools.product([0, 1], repeat=3)]
check(AND3, AND3_TABLE, "three-input AND by hand")

## Task 1.6 — XOR

The best setting found for XOR, and a record of the attempt.

In [ ]:
XOR_TRY = ThresholdUnit((..., ...), ...)

TABLES["XOR"] = [(0, 0, 0), (0, 1, 1), (1, 0, 1), (1, 1, 0)]
correct = sum(XOR_TRY(a, b) == expected for a, b, expected in TABLES["XOR"])
print(f"{correct} of 4 rows correct with w={XOR_TRY.w}, theta={XOR_TRY.theta}")

## Reflection — Task 1

- Which part took longest, and why?
- What did you get wrong first, and how did you find out?
- Which of 1.1, 1.3 and 1.4 could you do again right now, without looking anything up?